# Gemini Annotation Pipeline: Variance & Agreement Experiment (8-Class Prompting -> 5-Class Merged Evaluation)

This notebook evaluates the stability, inter-run agreement, and metric variance of the **Gemini Annotation Pipeline (Few-Shot Prompting Strategy)** across **5 repeated runs**.

### Experiment Setup:
1. **Prompting Schema**: Gemini is prompted using the full **8-class taxonomy** (`Background`, `Achievements`, `Education`, `Work Experience`, `Interests`, `Motivators`, `Learnings`, `Others`).
2. **Evaluation Merging**: Predictions and ground-truth human labels are merged into the **5-class target schema** (`Motivators`, `Interests`, `Learnings`, `Others` $\\rightarrow$ `Others`).
3. **Dataset Sampling**: 10 random documents (`seed = 42`) sampled from `data/human_dataset.csv` (totaling 703 sentences across document IDs: `[13, 17, 19, 25, 26, 30, 32, 39, 45, 48]`).
4. **Execution**: 5 independent runs of the few-shot prompting strategy on the sampled 703 sentences.
5. **Inter-Run Agreement Analysis**:
   - **Aggregate Agreement**: Exact Multi-Label Consensus (100% match across 5 runs on merged labels), Average Pairwise Jaccard Similarity, and Overall Fleiss' Kappa.
   - **Per-Class Agreement**: 100% Consensus Rate and Fleiss' Kappa per merged target category across the 5 runs.
6. **Evaluation Metrics & Variance**:
   - Compute Subset Accuracy (Exact Match), Precision, Recall, and F1-Score (micro, macro, weighted, samples, and per-class) for each run against merged human ground truth.
   - Calculate and report the **Mean**, **Range** (max - min), and **Variance** (sample variance) across the 5 runs.


In [1]:
import os
import ast
import json
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import statsmodels.stats.inter_rater as ir
import IPython.display as display

# Import workspace pipeline modules & taxonomy configs
import src.config as config
from src.prompt_builder import build_batch_classification_prompt
from src.gemini_client import get_batch_classification
from src.config import (
    CATEGORIES,
    DEFINITIONS,
    FEW_SHOT_EXAMPLES,
    CATEGORIES_5CLASS, 
    DEFINITIONS_5CLASS, 
    FEW_SHOT_EXAMPLES_5CLASS, 
    BATCH_SIZE
)

MERGE_MAP = {
    'background': 'Background',
    'achievements': 'Achievements',
    'education': 'Education',
    'work experience': 'Work Experience',
    'interests': 'Others',
    'motivators': 'Others',
    'learnings': 'Others',
    'others': 'Others'
}

def merge_ground_truth(lbl_list):
    """
    Maps ground truth human labels (Motivators, Interests, Learnings -> Others).
    """
    if isinstance(lbl_list, str):
        try:
            lbl_list = ast.literal_eval(lbl_list)
        except:
            lbl_list = [lbl_list]
            
    mapped = [MERGE_MAP.get(str(x).strip().lower(), 'Others') for x in lbl_list]
    return list(dict.fromkeys(mapped))

print("Setup complete.")
print(f"Prompting Schema Categories (8-Class): {CATEGORIES}")
print(f"Evaluation Schema Target Categories (5-Class Merged): {CATEGORIES_5CLASS}")


/home/ishan07/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Loaded 6 contrastive examples from data/conflicting_pairs_with_rationales.csv
Setup complete.
Prompting Schema Categories (8-Class): ['Background', 'Achievements', 'Education', 'Work Experience', 'Interests', 'Motivators', 'Learnings', 'Others']
Evaluation Schema Target Categories (5-Class Merged): ['Background', 'Achievements', 'Education', 'Work Experience', 'Others']


In [2]:
# 1. Load Human Ground-Truth Dataset and Sample 10 Random Docs (Seed 42)
HUMAN_DATASET_PATH = 'data/human_dataset.csv'
df_human_full = pd.read_csv(HUMAN_DATASET_PATH)

# Seed 42 document sampling
unique_docs = sorted(df_human_full['doc_id'].unique())
np.random.seed(42)
sampled_docs = np.random.choice(unique_docs, size=10, replace=False)

df_human = df_human_full[df_human_full['doc_id'].isin(sampled_docs)].copy().reset_index(drop=True)
df_human['raw_true_labels'] = df_human['labels'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_human['true_labels_5class'] = df_human['raw_true_labels'].apply(merge_ground_truth)

print(f"Sampled 10 random documents (Seed 42): {sorted(sampled_docs)}")
print(f"Total sentences extracted: {len(df_human)} (out of {len(df_human_full)} full dataset)")
df_human[['doc_id', 'sent_id', 'text', 'raw_true_labels', 'true_labels_5class']].head(10)


Sampled 10 random documents (Seed 42): [13, 17, 19, 25, 26, 30, 32, 39, 45, 48]
Total sentences extracted: 703 (out of 3661 full dataset)


,doc_id,sent_id,text,raw_true_labels,true_labels_5class
0,13,0,Donna J. Nelson (born 1954) is an American che...,"[Background, Work Experience]","[Background, Work Experience]"
1,13,1,"Nelson specializes in organic chemistry, which...",[Work Experience],[Work Experience]
2,13,2,Nelson served as the science advisor to the AM...,[Work Experience],[Work Experience]
3,13,3,She was the 2016 President of the American Che...,[Work Experience],[Work Experience]
4,13,4,Nelson's research focused on six primary topic...,[Work Experience],[Work Experience]
5,13,5,"Within Scientific Research, Nelson's topics ha...",[Work Experience],[Work Experience]
6,13,6,"Under America's Scientific Readiness, she focu...",[Work Experience],[Work Experience]
7,13,7,"Nelson was born in Eufaula, Oklahoma, a small ...",[Background],[Background]
8,13,8,Her father was the only physician in the town.,[Background],[Background]
9,13,9,She earned her Bachelor of Science degree in c...,[Education],[Education]


In [3]:
# 2. Pipeline Execution Engine (8-Class Few-Shot Prompting -> Merged 5-Class Evaluation)
def run_few_shot_pipeline(df_input, run_idx, batch_size=30, force_rerun=False):
    """
    Executes 8-Class Gemini classification on sampled dataset for a given run_idx,
    and stores both raw 8-class and merged 5-class predictions.
    """
    json_output_file = f'data/variance_fewshot_run_{run_idx}.json'
    csv_output_file = f'data/variance_fewshot_run_{run_idx}.csv'
    
    if not force_rerun and os.path.exists(json_output_file):
        print(f"Loading cached Run {run_idx} predictions from {json_output_file}...")
        with open(json_output_file, 'r', encoding='utf-8') as f:
            return json.load(f)
            
    original_strategy = config.PROMPTING_STRATEGY
    config.PROMPTING_STRATEGY = 'few_shot'
    
    items = []
    for idx, row in df_input.iterrows():
        items.append({
            'data': {
                'sent_id': int(row['sent_id']),
                'doc_id': int(row['doc_id']),
                'text': str(row['text']),
                'raw_true_labels': row['raw_true_labels'],
                'true_labels_5class': row['true_labels_5class'],
                'title': str(row['title'])
            },
            'temp_id': idx
        })
        
    def chunk_list(l, n):
        for i in range(0, len(l), n):
            yield l[i:i + n]
            
    chunks = list(chunk_list(items, batch_size))
    results_list = []
    previous_sentence = None
    
    print(f"Running 8-Class Gemini Few-Shot Pipeline (Run {run_idx}) on {len(items)} sentences ({len(chunks)} batches)...")
    
    for batch_idx, batch in tqdm(enumerate(chunks), total=len(chunks), desc=f"Run {run_idx}"):
        try:
            if batch[0]['data']['sent_id'] == 0:
                previous_sentence = None
                
            # Prompt builder with default 8-class definitions, categories, and exemplars
            prompt = build_batch_classification_prompt(
                batch, 
                previous_sentence=previous_sentence,
                categories=CATEGORIES,
                definitions=DEFINITIONS,
                few_shot_examples=FEW_SHOT_EXAMPLES
            )
            
            # Gemini API call enforcing 8-class JSON response schema
            batch_results = get_batch_classification(batch, prompt, categories=CATEGORIES)
            res_map = {str(r.get('id')): r.get('c', ['Others']) for r in batch_results if r.get('id') is not None}
            
            for item in batch:
                s_id = str(item['data']['sent_id'])
                pred_c = res_map.get(s_id, ['Others'])
                if isinstance(pred_c, str):
                    pred_c = [pred_c]
                item_res = dict(item)
                item_res['data']['pred_labels_raw'] = pred_c
                item_res['data']['pred_labels_5class'] = merge_ground_truth(pred_c)
                results_list.append(item_res)
                
            previous_sentence = batch[-1]['data']['text']
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Run {run_idx} Batch {batch_idx} error: {e}")
            for item in batch:
                item_res = dict(item)
                item_res['data']['pred_labels_raw'] = ['Others']
                item_res['data']['pred_labels_5class'] = ['Others']
                results_list.append(item_res)
                
    config.PROMPTING_STRATEGY = original_strategy
    
    # Save JSON
    if json_output_file:
        with open(json_output_file, 'w', encoding='utf-8') as f:
            json.dump(results_list, f, indent=2, ensure_ascii=False)
        print(f"[✓] Saved Run {run_idx} predictions JSON to {json_output_file}")
        
    # Save CSV
    if csv_output_file:
        export_rows = []
        for res in results_list:
            d = res['data']
            t_lbls_5 = d.get('true_labels_5class', [])
            p_lbls_raw = d.get('pred_labels_raw', [])
            p_lbls_5 = d.get('pred_labels_5class', [])
            export_rows.append({
                'doc_id': d.get('doc_id'),
                'sent_id': d.get('sent_id'),
                'title': d.get('title'),
                'text': d.get('text'),
                'raw_true_labels': json.dumps(d.get('raw_true_labels', [])),
                'true_labels_5class': json.dumps(t_lbls_5),
                'pred_labels_raw': json.dumps(p_lbls_raw),
                'pred_labels_5class': json.dumps(p_lbls_5),
                'exact_match_5class': set(t_lbls_5) == set(p_lbls_5)
            })
        pd.DataFrame(export_rows).to_csv(csv_output_file, index=False)
        print(f"[✓] Saved Run {run_idx} predictions CSV to {csv_output_file}")
        
    return results_list


In [4]:
# 3. Execute / Load All 5 Few-Shot Runs
FORCE_RERUN = False
NUM_RUNS = 5

all_runs_predictions = []
for run_i in range(1, NUM_RUNS + 1):
    run_preds = run_few_shot_pipeline(
        df_human,
        run_idx=run_i,
        batch_size=30,
        force_rerun=FORCE_RERUN
    )
    all_runs_predictions.append(run_preds)

print(f"\n[✓] Successfully collected {len(all_runs_predictions)} runs of predictions ({len(all_runs_predictions[0])} sentences each).")


Running 8-Class Gemini Few-Shot Pipeline (Run 1) on 703 sentences (24 batches)...


Run 1: 100%|██████████| 24/24 [21:02<00:00, 52.60s/it]


[✓] Saved Run 1 predictions JSON to data/variance_fewshot_run_1.json
[✓] Saved Run 1 predictions CSV to data/variance_fewshot_run_1.csv
Running 8-Class Gemini Few-Shot Pipeline (Run 2) on 703 sentences (24 batches)...


Run 2: 100%|██████████| 24/24 [18:40<00:00, 46.67s/it]


[✓] Saved Run 2 predictions JSON to data/variance_fewshot_run_2.json
[✓] Saved Run 2 predictions CSV to data/variance_fewshot_run_2.csv
Running 8-Class Gemini Few-Shot Pipeline (Run 3) on 703 sentences (24 batches)...


...

[✓] Saved Run 3 predictions JSON to data/variance_fewshot_run_3.json
[✓] Saved Run 3 predictions CSV to data/variance_fewshot_run_3.csv
Running 8-Class Gemini Few-Shot Pipeline (Run 4) on 703 sentences (24 batches)...


Run 4: 100%|██████████| 24/24 [20:40<00:00, 51.68s/it]


[✓] Saved Run 4 predictions JSON to data/variance_fewshot_run_4.json
[✓] Saved Run 4 predictions CSV to data/variance_fewshot_run_4.csv
Running 8-Class Gemini Few-Shot Pipeline (Run 5) on 703 sentences (24 batches)...


Run 5: 100%|██████████| 24/24 [19:56<00:00, 49.84s/it]

[✓] Saved Run 5 predictions JSON to data/variance_fewshot_run_5.json
[✓] Saved Run 5 predictions CSV to data/variance_fewshot_run_5.csv

[✓] Successfully collected 5 runs of predictions (703 sentences each).


In [5]:
# 4. Inter-Run Agreement Analysis (Aggregate and Per-Class on Merged 5-Class Schema)
mlb = MultiLabelBinarizer(classes=CATEGORIES_5CLASS)

# Extract binary prediction matrices for each run: List of shape (N, 5)
run_bin_matrices = []
for run_preds in all_runs_predictions:
    y_p = [item['data']['pred_labels_5class'] for item in run_preds]
    run_bin_matrices.append(mlb.fit_transform(y_p))

num_sentences = len(df_human)
num_classes = len(CATEGORIES_5CLASS)

# --- A. Aggregate Agreement ---
exact_consensus_count = 0
jaccard_scores = []

for s_idx in range(num_sentences):
    run_label_sets = [set(all_runs_predictions[r][s_idx]['data']['pred_labels_5class']) for r in range(NUM_RUNS)]
    
    if all(s == run_label_sets[0] for s in run_label_sets):
        exact_consensus_count += 1
        
    for r1 in range(NUM_RUNS):
        for r2 in range(r1 + 1, NUM_RUNS):
            union_len = len(run_label_sets[r1].union(run_label_sets[r2]))
            inter_len = len(run_label_sets[r1].intersection(run_label_sets[r2]))
            jaccard_scores.append(inter_len / union_len if union_len > 0 else 1.0)

exact_multilabel_agreement = exact_consensus_count / num_sentences
avg_pairwise_jaccard = np.mean(jaccard_scores)

overall_ratings = []
for s_idx in range(num_sentences):
    for c_idx in range(num_classes):
        votes_1 = sum(run_bin_matrices[r][s_idx, c_idx] for r in range(NUM_RUNS))
        votes_0 = NUM_RUNS - votes_1
        overall_ratings.append([votes_0, votes_1])

overall_fleiss_kappa = ir.fleiss_kappa(np.array(overall_ratings))

aggregate_agreement_df = pd.DataFrame([
    {'Metric': 'Exact Multi-Label Consensus (100% Match across 5 Runs)', 'Value': exact_multilabel_agreement, 'Formatted': f"{exact_multilabel_agreement * 100:.2f}%"},
    {'Metric': 'Average Pairwise Jaccard Similarity', 'Value': avg_pairwise_jaccard, 'Formatted': f"{avg_pairwise_jaccard * 100:.2f}%"},
    {'Metric': 'Overall Fleiss\' Kappa', 'Value': overall_fleiss_kappa, 'Formatted': f"{overall_fleiss_kappa:.4f}"}
])

print('=' * 75)
print('      AGGREGATE INTER-RUN AGREEMENT (5 REPEATED RUNS - MERGED 5-CLASS)')
print('=' * 75)
try:
    print(aggregate_agreement_df[['Metric', 'Formatted']].to_markdown(index=False))
except Exception:
    print(aggregate_agreement_df[['Metric', 'Formatted']].to_string(index=False))
print('=' * 75)

# --- B. Per-Class Agreement ---
per_class_agreement = []
for c_idx, cat in enumerate(CATEGORIES_5CLASS):
    class_consensus_count = 0
    class_ratings = []
    
    for s_idx in range(num_sentences):
        votes_1 = sum(run_bin_matrices[r][s_idx, c_idx] for r in range(NUM_RUNS))
        votes_0 = NUM_RUNS - votes_1
        class_ratings.append([votes_0, votes_1])
        if votes_1 == 0 or votes_1 == NUM_RUNS:
            class_consensus_count += 1
            
    consensus_rate = class_consensus_count / num_sentences
    cat_kappa = ir.fleiss_kappa(np.array(class_ratings))
    
    per_class_agreement.append({
        'Category': cat,
        '100% Consensus Rate': f"{consensus_rate * 100:.2f}%",
        'Fleiss\' Kappa': f"{cat_kappa:.4f}"
    })

per_class_agreement_df = pd.DataFrame(per_class_agreement).set_index('Category')

print('\n' + '=' * 75)
print('        PER-CLASS INTER-RUN AGREEMENT BREAKDOWN (MERGED 5-CLASS)')
print('=' * 75)
try:
    print(per_class_agreement_df.to_markdown())
except Exception:
    print(per_class_agreement_df.to_string())
print('=' * 75)


      AGGREGATE INTER-RUN AGREEMENT (5 REPEATED RUNS - MERGED 5-CLASS)
                                                Metric Formatted
Exact Multi-Label Consensus (100% Match across 5 Runs)    78.09%
                   Average Pairwise Jaccard Similarity    90.87%
                                 Overall Fleiss' Kappa    0.8875

        PER-CLASS INTER-RUN AGREEMENT BREAKDOWN (MERGED 5-CLASS)
                100% Consensus Rate Fleiss' Kappa
Category                                         
Background                   98.15%        0.8537
Achievements                 93.60%        0.7770
Education                    99.29%        0.9465
Work Experience              85.49%        0.8615
Others                       84.07%        0.8422


In [6]:
# 5. Evaluation Metrics across 5 Runs: Mean, Range, and Variance (Merged 5-Class Schema)
y_true_5class = [item['data']['true_labels_5class'] for item in all_runs_predictions[0]]
y_true_bin = mlb.fit_transform(y_true_5class)

run_metrics_list = []
run_per_class_list = []

metric_names = [
    'Subset Accuracy (Exact Match)',
    'Precision (Micro)', 'Recall (Micro)', 'F1-Score (Micro)',
    'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)',
    'Precision (Weighted)', 'Recall (Weighted)', 'F1-Score (Weighted)',
    'Precision (Samples)', 'Recall (Samples)', 'F1-Score (Samples)'
]

for r_idx, run_preds in enumerate(all_runs_predictions):
    y_pred_5class = [item['data']['pred_labels_5class'] for item in run_preds]
    y_pred_bin = mlb.transform(y_pred_5class)
    
    m_dict = {
        'Subset Accuracy (Exact Match)': accuracy_score(y_true_bin, y_pred_bin)
    }
    for avg in ['micro', 'macro', 'weighted', 'samples']:
        m_dict[f'Precision ({avg.capitalize()})'] = precision_score(y_true_bin, y_pred_bin, average=avg, zero_division=0)
        m_dict[f'Recall ({avg.capitalize()})'] = recall_score(y_true_bin, y_pred_bin, average=avg, zero_division=0)
        m_dict[f'F1-Score ({avg.capitalize()})'] = f1_score(y_true_bin, y_pred_bin, average=avg, zero_division=0)
        
    run_metrics_list.append(m_dict)
    
    p_cat = precision_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
    r_cat = recall_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
    f1_cat = f1_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
    
    cat_df = pd.DataFrame({
        'Category': CATEGORIES_5CLASS,
        'Precision': p_cat,
        'Recall': r_cat,
        'F1-Score': f1_cat
    }).set_index('Category')
    run_per_class_list.append(cat_df)

# --- Compute Overall Metrics Statistics ---
stats_rows = []
for m in metric_names:
    scores = np.array([run_m[m] for run_m in run_metrics_list])
    m_mean = np.mean(scores)
    m_range = np.ptp(scores)
    m_var = np.var(scores, ddof=1) if len(scores) > 1 else 0.0
    m_std = np.std(scores, ddof=1) if len(scores) > 1 else 0.0
    
    stats_rows.append({
        'Metric': m,
        'Mean': m_mean,
        'Range': m_range,
        'Variance': m_var,
        'Std Dev': m_std,
        'Mean (%)': f"{m_mean * 100:.2f}%",
        'Range (%)': f"{m_range * 100:.2f}%",
        'Variance': f"{m_var:.6f}"
    })

eval_stats_df = pd.DataFrame(stats_rows)

print('=' * 85)
print('    EVALUATION METRICS VARIANCE SUMMARY (5 FEW-SHOT RUNS - MERGED 5-CLASS)')
print('=' * 85)
try:
    print(eval_stats_df[['Metric', 'Mean (%)', 'Range (%)', 'Variance']].to_markdown(index=False))
except Exception:
    print(eval_stats_df[['Metric', 'Mean (%)', 'Range (%)', 'Variance']].to_string(index=False))
print('=' * 85)

display.display(eval_stats_df[['Metric', 'Mean', 'Range', 'Variance', 'Std Dev', 'Mean (%)', 'Range (%)']]
                .style.format({'Mean': '{:.4f}', 'Range': '{:.4f}', 'Variance': '{:.6f}', 'Std Dev': '{:.4f}'})
                .set_caption('Evaluation Metrics Mean, Range, and Variance across 5 Runs (Merged 5-Class Schema)'))


    EVALUATION METRICS VARIANCE SUMMARY (5 FEW-SHOT RUNS - MERGED 5-CLASS)
                       Metric Mean (%) Range (%) Variance
Subset Accuracy (Exact Match)   81.45%     1.42% 0.000038
            Precision (Micro)   83.09%     1.69% 0.000039
               Recall (Micro)   85.02%     2.47% 0.000082
             F1-Score (Micro)   84.04%     1.53% 0.000032
            Precision (Macro)   81.53%     3.59% 0.000210
               Recall (Macro)   86.66%     2.97% 0.000161
             F1-Score (Macro)   82.65%     2.90% 0.000176
         Precision (Weighted)   84.52%     1.73% 0.000048
            Recall (Weighted)   85.02%     2.47% 0.000082
          F1-Score (Weighted)   84.17%     1.82% 0.000043
          Precision (Samples)   84.31%     1.42% 0.000030
             Recall (Samples)   85.69%     2.35% 0.000071
           F1-Score (Samples)   84.60%     1.62% 0.000036


ValueError: Unknown format code 'f' for object of type 'str'

In [7]:
# 6. Per-Class Evaluation Metrics Variance Summary (Merged 5-Class Schema)
per_class_stats = []

for cat in CATEGORIES_5CLASS:
    p_scores = np.array([run_per_class_list[r].loc[cat, 'Precision'] for r in range(NUM_RUNS)])
    r_scores = np.array([run_per_class_list[r].loc[cat, 'Recall'] for r in range(NUM_RUNS)])
    f1_scores = np.array([run_per_class_list[r].loc[cat, 'F1-Score'] for r in range(NUM_RUNS)])
    
    per_class_stats.append({
        'Category': cat,
        'Precision Mean': f"{np.mean(p_scores) * 100:.2f}%",
        'Precision Range': f"{np.ptp(p_scores) * 100:.2f}%",
        'Precision Var': f"{np.var(p_scores, ddof=1):.6f}",
        'Recall Mean': f"{np.mean(r_scores) * 100:.2f}%",
        'Recall Range': f"{np.ptp(r_scores) * 100:.2f}%",
        'Recall Var': f"{np.var(r_scores, ddof=1):.6f}",
        'F1 Mean': f"{np.mean(f1_scores) * 100:.2f}%",
        'F1 Range': f"{np.ptp(f1_scores) * 100:.2f}%",
        'F1 Var': f"{np.var(f1_scores, ddof=1):.6f}"
    })

per_class_stats_df = pd.DataFrame(per_class_stats).set_index('Category')

print('\n' + '=' * 85)
print('    PER-CLASS EVALUATION METRICS VARIANCE BREAKDOWN (MERGED 5-CLASS)')
print('=' * 85)
try:
    print(per_class_stats_df.to_markdown())
except Exception:
    print(per_class_stats_df.to_string())
print('=' * 85)



    PER-CLASS EVALUATION METRICS VARIANCE BREAKDOWN (MERGED 5-CLASS)
                Precision Mean Precision Range Precision Var Recall Mean Recall Range Recall Var F1 Mean F1 Range    F1 Var
Category                                                                                                                   
Background              97.39%           4.55%      0.000570      75.86%       13.79%   0.002973  85.19%    8.93%  0.001313
Achievements            55.29%          16.38%      0.003601      91.61%        3.23%   0.000312  68.85%   13.79%  0.002630
Education               84.24%          10.90%      0.001984      96.36%        4.55%   0.000413  89.88%    8.15%  0.001130
Work Experience         81.32%           2.46%      0.000085      91.85%        4.94%   0.000408  86.26%    2.78%  0.000104
Others                  89.40%           4.12%      0.000279      77.59%        3.41%   0.000147  83.06%    2.64%  0.000095
